In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score
)

# 1. PREPARAÇÃO E GERAÇÃO DE DADOS SINTÉTICOS

In [3]:
# ==========================================
# 1. PREPARAÇÃO E GERAÇÃO DE DADOS SINTÉTICOS
# ==========================================
np.random.seed(42)
n_samples = 1000

# Variáveis preditoras
frequencia = np.random.uniform(50, 100, n_samples)          # Frequência (%)
media_notas = np.random.uniform(2.0, 10.0, n_samples)       # Média escolar (0-10)
reprovacoes = np.random.poisson(0.5, n_samples)             # Histórico de reprovações
renda_familiar = np.random.exponential(2.5, n_samples) + 1 # Renda em salários mínimos
trabalha = np.random.binomial(1, 0.3, n_samples)            # 1 = Trabalha, 0 = Não trabalha
escolaridade_mae = np.random.randint(0, 4, n_samples)       # 0: Fundamental Incompleto a 3: Superior

# Cálculo da probabilidade teórica de evasão (Log-Odds)
# Nota: Frequência, notas, renda e escolaridade reduzem o risco; reprovações e trabalho aumentam.
log_odds = (
    1.5 
    - 0.08 * frequencia 
    - 0.45 * media_notas 
    + 0.85 * reprovacoes 
    - 0.30 * renda_familiar 
    + 0.60 * trabalha 
    - 0.25 * escolaridade_mae
)
probabilidade = 1 / (1 + np.exp(-log_odds))
evasao = np.random.binomial(1, probabilidade)

df = pd.DataFrame({
    'frequencia': frequencia,
    'media_notas': media_notas,
    'reprovacoes': reprovacoes,
    'renda_familiar': renda_familiar,
    'trabalha': trabalha,
    'escolaridade_mae': escolaridade_mae,
    'evasao': evasao
})

# 2. ANÁLISE EXPLORATÓRIA INICIAL

In [4]:
# ==========================================
# 2. ANÁLISE EXPLORATÓRIA INICIAL
# ==========================================
print("--- DISTRIBUIÇÃO DA VARIÁVEL ALVO (EVASÃO) ---")
print(df['evasao'].value_counts(normalize=True).rename({0: 'Permanência (0)', 1: 'Evasão (1)'}))
print("\n--- ESTATÍSTICAS DESCRITIVAS ---")
print(df.describe().T[['mean', 'std', 'min', '50%', 'max']])

--- DISTRIBUIÇÃO DA VARIÁVEL ALVO (EVASÃO) ---
Permanência (0)    0.998
Evasão (1)         0.002
Name: evasao, dtype: float64

--- ESTATÍSTICAS DESCRITIVAS ---
                       mean        std        min        50%        max
frequencia        74.512828  14.606868  50.231601  74.840369  99.985884
media_notas        6.056138   2.337519   2.025746   6.149871   9.995310
reprovacoes        0.510000   0.720013   0.000000   0.000000   4.000000
renda_familiar     3.365567   2.414837   1.000077   2.620887  20.308824
trabalha           0.298000   0.457608   0.000000   0.000000   1.000000
escolaridade_mae   1.416000   1.094597   0.000000   1.000000   3.000000
evasao             0.002000   0.044699   0.000000   0.000000   1.000000


# 3. DIVISÃO E PADRONIZAÇÃO DOS DADOS

In [8]:
# ==========================================
# 3. DIVISÃO E PADRONIZAÇÃO DOS DADOS
# ==========================================
X = df.drop(columns=['evasao'])
y = df['evasao']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Converte de volta para DataFrame mantendo os nomes das colunas
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

# 4. AVALIAÇÃO DE SIGNIFICÂNCIA (STATSMODELS)

In [9]:
# ==========================================
# 4. AVALIAÇÃO DE SIGNIFICÂNCIA (STATSMODELS)
# ==========================================
X_train_sm = sm.add_constant(X_train_scaled)
sm_model = sm.Logit(y_train, X_train_sm).fit(disp=False)

print("\n--- SUMÁRIO ESTATÍSTICO (STATSMODELS) ---")
summary_df = pd.DataFrame({
    'Coeficiente': sm_model.params[1:],
    'P-Value': sm_model.pvalues[1:],
    'Odds Ratio (OR)': np.exp(sm_model.params[1:])
}, index=X.columns)
print(summary_df)

C:\Users\89821491120\AppData\Local\anaconda3\lib\site-packages\statsmodels\discrete\discrete_model.py:1819: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\89821491120\AppData\Local\anaconda3\lib\site-packages\statsmodels\discrete\discrete_model.py:1872: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q*np.dot(X,params))))


LinAlgError: Singular matrix

In [10]:
# ==========================================
# 5. TREINAMENTO E MODELAGEM (SCIKIT-LEARN)
# ==========================================
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Previsões
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

In [11]:
# ==========================================
# 6. AVALIAÇÃO DE DESEMPENHO
# ==========================================
print("\n--- MÉTRICAS DE DESEMPENHO (CONJUNTO DE TESTE) ---")
print(f"Acurácia:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precisão:  {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")

print("\n--- MATRIZ DE CONFUSÃO ---")
cm = confusion_matrix(y_test, y_pred)
print(f"Verdadeiros Negativos (Permanência Correta): {cm[0,0]}")
print(f"Falsos Positivos (Alerta Falso):            {cm[0,1]}")
print(f"Falsos Negativos (Evasão Não Detectada):     {cm[1,0]}")
print(f"Verdadeiros Positivos (Evasão Detectada):    {cm[1,1]}")


--- MÉTRICAS DE DESEMPENHO (CONJUNTO DE TESTE) ---
Acurácia:  0.9967
Precisão:  0.0000
Recall:    0.0000
F1-Score:  0.0000
ROC-AUC:   0.9866

--- MATRIZ DE CONFUSÃO ---
Verdadeiros Negativos (Permanência Correta): 299
Falsos Positivos (Alerta Falso):            0
Falsos Negativos (Evasão Não Detectada):     1
Verdadeiros Positivos (Evasão Detectada):    0


C:\Users\89821491120\AppData\Local\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
